In [5]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

# read_csv() 함수로 df 생성
df = pd.read_csv('./data/auto-mpg.csv', header=None)

# 열 이름을 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name'] 

# horsepower 열의 누락 데이터('?') 삭제하고 실수형으로 변환
df['horsepower'] = df['horsepower'].replace('?', np.nan)      # '?'을 np.nan으로 변경
df = df.dropna(subset=['horsepower'], axis=0)                 # 누락데이터 행을 삭제
df['horsepower'] = df['horsepower'].astype('float')           # 문자열을 실수형으로 변환

# np.histogram 으로 3개의 bin으로 나누는 경계 값의 리스트 구하기
count, bin_dividers = np.histogram(df['horsepower'], bins=3)

# 3개의 bin에 이름 지정
bin_names = ['저출력', '보통출력', '고출력']

# pd.cut 으로 각 데이터를 3개의 bin에 할당
df['hp_bin'] = pd.cut(x=df['horsepower'],     # 데이터 배열
                      bins=bin_dividers,      # 경계 값 리스트
                      labels=bin_names,       # bin 이름
                      include_lowest=True)    # 첫 경계값 포함

# sklern 라이브러리 불러오기
from sklearn import preprocessing    

# 전처리를 위한 encoder 객체 만들기
label_encoder = preprocessing.LabelEncoder()       # label encoder 생성
onehot_encoder = preprocessing.OneHotEncoder()     # one hot encoder 생성

# label encoder로 문자열 범주를 숫자형 범주로 변환
onehot_labeled = label_encoder.fit_transform(df['hp_bin'].head(15))  
print(onehot_labeled)
print(type(onehot_labeled))

# 2차원 행렬로 형태 변경
onehot_reshaped = onehot_labeled.reshape(len(onehot_labeled), 1) 
print(onehot_reshaped)
print(type(onehot_reshaped))

# 희소행렬로 변환
onehot_fitted = onehot_encoder.fit_transform(onehot_reshaped)
print(onehot_fitted)
print(type(onehot_fitted))

[1 1 1 1 1 0 0 0 0 0 0 1 1 0 2]
<class 'numpy.ndarray'>
[[1]
 [1]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [0]
 [2]]
<class 'numpy.ndarray'>
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (15, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 1)	1.0
  (2, 1)	1.0
  (3, 1)	1.0
  (4, 1)	1.0
  (5, 0)	1.0
  (6, 0)	1.0
  (7, 0)	1.0
  (8, 0)	1.0
  (9, 0)	1.0
  (10, 0)	1.0
  (11, 1)	1.0
  (12, 1)	1.0
  (13, 0)	1.0
  (14, 2)	1.0
<class 'scipy.sparse._csr.csr_matrix'>


In [6]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [7]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

# read_csv() 함수로 df 생성
df = pd.read_csv('./data/auto-mpg.csv', header=None)

# 열 이름을 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


In [8]:
# 연비를 3개 구간으로 나눠서 새로운 컬럼명으로 추가
# 저연비, 보통, 고연비
# 연속형(수치형) -> 범주형(category)형 분리
df.mpg
df['mpg_qcut'] =  pd.qcut(df.mpg,q=3,labels=['저연비', '보통', '고연비'])
print(df.mpg_qcut.value_counts())
# 계급구간확인  라벨없이 적용후
temp = pd.qcut(df.mpg,q=3)
print(f'구간별 데이터 범위 : {temp.cat.categories}')

mpg_qcut
저연비    143
고연비    133
보통     122
Name: count, dtype: int64
구간별 데이터 범위 : IntervalIndex([(8.999, 19.0], (19.0, 26.933], (26.933, 46.6]], dtype='interval[float64, right]')


In [9]:
size, bins_range =  np.histogram(df.mpg,bins=np.array(3))
df['mpg_cut'] = pd.cut(df.mpg,bins=bins_range,labels=['저연비', '보통', '고연비'])
df.mpg_cut.value_counts()
print(f'계급별 구간 : {bins_range}')

계급별 구간 : [ 9.         21.53333333 34.06666667 46.6       ]


In [10]:
pd.get_dummies(df.mpg_qcut,dtype=float)

,저연비,보통,고연비
0,1.0,0.0,0.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
...,...,...,...
393,0.0,0.0,1.0
394,0.0,0.0,1.0
395,0.0,0.0,1.0
396,0.0,0.0,1.0


In [11]:
pd.get_dummies(df.mpg_cut,dtype=float)

,저연비,보통,고연비
0,1.0,0.0,0.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
...,...,...,...
393,0.0,1.0,0.0
394,0.0,0.0,1.0
395,0.0,1.0,0.0
396,0.0,1.0,0.0


In [12]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [13]:
# 다중 공선성 : 회귀분석(여러변수로 결과를 예측하는 모델), 독립변수(컬럼들..)끼리
# 서로 너무 강하게 상관관계를 가질때, 문제가 생긴다.
# 독립변수끼리의 상관계수가 높으면 문제가 생긴다.

# 다중 공선성 ^
# 회귀분석에서 독립변수끼리 너무 강하게 상관되어 있는 상태
# 독립변수가 서로 강하게 상관되면 다음과 같은 문제가 발생합니다:

# 회귀계수(β)의 신뢰성 감소
# 계수가 실제로 어떤 독립변수의 효과인지 구분하기 어려워집니다.
# 작은 변화에도 계수가 크게 변할 수 있어 불안정한 추정치가 나옵니다.

# 통계적 유의성 판단이 어려움
# 표준오차(Standard Error)가 커져서 t-값이 작아집니다.
# → 결국 중요한 변수도 통계적으로 유의하지 않다고 판단될 수 있습니다.

# 모델 해석의 어려움
# 독립변수들 간의 영향이 섞이므로 개별 변수의 기여도 해석이 어렵습니다.

# 예측 성능 자체는 크게 영향을 안 줄 수도 있음
# 다중공선성은 계수 추정과 해석에 문제를 주지만,
# 예측(Prediction) 정확도에는 바로 큰 영향을 주지 않을 수도 있습니다.
# → 다만 변수 선택, 해석, 정책 결정에는 문제가 됩니다.

# https://chatgpt.com/share/68ddd165-28d4-8012-9c56-44066da90591

In [14]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

# read_csv() 함수로 df 생성
df = pd.read_csv('./data/auto-mpg.csv', header=None)

# 열 이름을 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

# 범주형 데이터 변환 및 생성
df['mpg_qcut'] =  pd.qcut(df.mpg,q=3,labels=['저연비', '보통', '고연비'])

from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False) # 객체 OneHotEncoder 클래스 부여
# https://chatgpt.com/share/68ddde12-34ac-8012-8760-33eec01b989c
# sklearn 계열은 사용방법이 통일
# fit : 적용
# transform : 변환
# fit_transform : 두개를 한꺼번에 실행
# 이런 형식으로 명령어가 구성이 되어있다.

temp = encoder.fit_transform(df[['mpg_qcut']]) # <- 2차원 데이터가 와야 한다
#“2차원 데이터가 와야 한다”는 조건은 sklearn의 OneHotEncoder.fit_transform() 메서드 때문에 필요합니다.
# df[['mpg_qcut']]처럼 DataFrame으로 감싸서 2차원으로 만들어주면 정상 동작합니다.
# 반대로 1차원 Series(df['mpg_qcut'])를 넣으면 에러가 납니다.

cols = encoder.get_feature_names_out(['mpg_qcut'])
pd.DataFrame(temp, columns=cols)
pd.concat([df.drop(columns=['mpg_qcut']), pd.DataFrame(temp, columns=cols)],axis=1)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,name,mpg_qcut_고연비,mpg_qcut_보통,mpg_qcut_저연비
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu,0.0,0.0,1.0
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320,0.0,0.0,1.0
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite,0.0,0.0,1.0
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst,0.0,0.0,1.0
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.00,2790.0,15.6,82,1,ford mustang gl,1.0,0.0,0.0
394,44.0,4,97.0,52.00,2130.0,24.6,82,2,vw pickup,1.0,0.0,0.0
395,32.0,4,135.0,84.00,2295.0,11.6,82,1,dodge rampage,1.0,0.0,0.0
396,28.0,4,120.0,79.00,2625.0,18.6,82,1,ford ranger,1.0,0.0,0.0


In [15]:
# get_dummy() : 빠르게 확인, 컬럼명 유지 pandas Dataframe / Series
    # 데이터 탐색, 시각화, 작은 데이터 셋

# OneHotEncoder() numpy array, DataFrame
    # 머신러닝 파이라인 / 모델 학습
    # fit , transform -> 학습 - 예측 데이터 일관성 유지, 큰데이터 셋

In [16]:
train_df = df.drop(columns=['mpg'])
train_df.head()
# cylinders, model year, origin --> onehot으로 범주형 데이터 변환
# 제조사 컬럼에서 제조사만 추출해서 -> onehot으로 범주형 데이터 변환
# 하나의 데이터 프레임으로 결합 concat 사용

# 강사 / 학생 풀이 비교
# https://chatgpt.com/s/t_68ddeab6212c81919b8c3e4cb75b1bf9
# 코드 길이 / 확장성 / 메모리 효율(pandas 친화성과 연관) / 실무 활용

,cylinders,displacement,horsepower,weight,acceleration,model year,origin,name,mpg_qcut
0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu,저연비
1,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320,저연비
2,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite,저연비
3,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst,저연비
4,8,302.0,140.0,3449.0,10.5,70,1,ford torino,저연비


In [17]:
print(train_df['cylinders'].unique())
print(train_df['model year'].unique())
print(train_df['origin'].unique())

[8 4 6 3 5]
[70 71 72 73 74 75 76 77 78 79 80 81 82]
[1 3 2]


In [ ]:
# 강사님 풀이
from sklearn.preprocessing import OneHotEncoder
# 도구(객체) 생성
encoder = OneHotEncoder()

# 컬럼명 지정 (1, 0, 1 이런식으로 했을때, 위에 달릴 컬럼명)
origin_cols = ['cylinders','model year', 'origin','maker']

# 제조사 컬럼에서 제조사만 추출
train_df['maker'] = [n_list[0] for n_list in df['name'].str.split()]
# 여기서 str을 사용하는 이유
# 시리즈에는 문자열 메서드인 split을 사용할 수 없다. 
# Series 안의 각 원소는 문자열이니까, 문자열 메서드를 적용할게라는 의미로 str을 붙인다.
# .str 메서드 작동동작을 보자면,
# 내부적으로 isinstance(x, str) 같은 체크를 하고, 아니면 결측치 처리해버린다.
# 문자열이 아닌 원소는 결측치처리를 해버린다는 말이니, 사용할때 안에 문자열 외의 타입이 있는지 유의해야한다.

total_onehots = []
for colname in origin_cols:
    encoder_fit_transform = encoder.fit_transform(train_df[[colname]])
    # scipy.sparse.csr_matrix (희소행렬)형태의 값이 encoder_fit_transform에 부여
    # https://chatgpt.com/s/t_68de543e6fcc81919f1e11353c2e3929
    cols = encoder.get_feature_names_out([colname])
    total_onehots.append(pd.DataFrame.sparse.from_spmatrix(encoder_fit_transform,columns=cols))
    # from_spmatrix() : sparse matrix를 DataFrame으로 바꿔주고 컬럼 이름 지정까지 해주는 함수

In [19]:
total_onehots.insert(0,train_df)
new_train_df = pd.concat(total_onehots,axis=1)
new_train_df = new_train_df.drop(columns=origin_cols)
new_train_df.head()

,displacement,horsepower,weight,acceleration,name,mpg_qcut,cylinders_3,cylinders_4,cylinders_5,cylinders_6,...,maker_renault,maker_saab,maker_subaru,maker_toyota,maker_toyouta,maker_triumph,maker_vokswagen,maker_volkswagen,maker_volvo,maker_vw
0,307.0,130.0,3504.0,12.0,chevrolet chevelle malibu,저연비,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,350.0,165.0,3693.0,11.5,buick skylark 320,저연비,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,318.0,150.0,3436.0,11.0,plymouth satellite,저연비,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,304.0,150.0,3433.0,12.0,amc rebel sst,저연비,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,302.0,140.0,3449.0,10.5,ford torino,저연비,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
# 학생 풀이
# cylinders, model year, origin이 변환이 안되었다.^ 그게 아니라 기존에 있는 컬럼을 삭제안해서 안보였다.
# 컬럼명 적절하게 다시 짓고 그걸 설정하는법 ^
# 좀 더 간략하고 간단 정확하게 하는 법, 추려보기?

# cylinders, model year, origin --> onehot으로 범주형 데이터 변환
# 제조사 컬럼에서 제조사만 추출해서 -> onehot으로 범주형 데이터 변환
# 하나의 데이터 프레임으로 결합 concat 사용

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
en_1 = OneHotEncoder(sparse_output=False)
en_2 = OneHotEncoder(sparse_output=False)
en_3 = OneHotEncoder(sparse_output=False)

train_df['cylinders'] = train_df['cylinders'].astype(str)
train_df['model year'] = train_df['model year'].astype(str)
train_df['origin'] = train_df['origin'].astype(str)

temp_1 = en_1.fit_transform(train_df[['cylinders']])
temp_2 = en_2.fit_transform(train_df[['model year']])
temp_3 = en_3.fit_transform(train_df[['origin']])

cols_1 = en_1.get_feature_names_out(['cylinders'])
cols_2 = en_2.get_feature_names_out(['model year'])
cols_3 = en_3.get_feature_names_out(['origin'])
# pd.DataFrame(temp, columns=cols)

# 앞 글자만 ' '를 기준으로 split 해서 제조사 뽑고 컬럼 만들기
train_df['abbr'] = train_df['name'].apply(lambda x: ''.join([word[0] for word in x.split()]))
# print(train_df)

# 해당 컬럼데이터를 범주형 데이터로 만들기
en_4 = OneHotEncoder(sparse_output=False)

temp_4 = en_4.fit_transform(train_df[['abbr']])

cols_4 = en_4.get_feature_names_out(['abbr'])

# 하나의 데이터 프레임으로 결합
df_1 = pd.DataFrame(temp_1, columns=cols_1, index=train_df.index)
df_2 = pd.DataFrame(temp_2, columns=cols_2, index=train_df.index)
df_3 = pd.DataFrame(temp_3, columns=cols_3, index=train_df.index)
df_4 = pd.DataFrame(temp_4, columns=cols_4, index=train_df.index)
# index = train_df.index

# concat 사용으로 총합
train_df2 = pd.concat([train_df, df_1, df_2, df_3, df_4], axis=1)

train_df2 = train_df2.drop(columns='origin')
train_df2 = train_df2.drop(columns='cylinders')
train_df2 = train_df2.drop(columns='model year')
train_df2.head()

,displacement,horsepower,weight,acceleration,name,mpg_qcut,maker,abbr,cylinders_3,cylinders_4,...,abbr_vm1,abbr_vp,abbr_vr,abbr_vrc,abbr_vrc(,abbr_vrcd,abbr_vrl,abbr_vs,abbr_vsb,abbr_vt3
0,307.0,130.0,3504.0,12.0,chevrolet chevelle malibu,저연비,chevrolet,ccm,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,350.0,165.0,3693.0,11.5,buick skylark 320,저연비,buick,bs3,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,318.0,150.0,3436.0,11.0,plymouth satellite,저연비,plymouth,ps,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,304.0,150.0,3433.0,12.0,amc rebel sst,저연비,amc,ars,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,302.0,140.0,3449.0,10.5,ford torino,저연비,ford,ft,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression # 선형 회귀 2
# 작은 : 0 / 보통 : 1 / 크 : 2
data = {
    'size' : [0,0,0,1,1,1,2,2,2],
    'height' : [120,121,119,122,123,121,125,124,126]
}
df = pd.DataFrame(data)

print(df)
# 실제 평균
df.groupby('size')['height'].mean()

   size  height
0     0     120
1     0     121
2     0     119
3     1     122
4     1     123
5     1     121
6     2     125
7     2     124
8     2     126


size
0    120.0
1    122.0
2    125.0
Name: height, dtype: float64

In [22]:
# 변환이나 따로 전처리 하지않고 학습하고 예측
model_lr = LinearRegression()
X = df.drop(columns = ['height'])
y = df['height']
X.shape, y.shape, type(y)
model_ly = LinearRegression()
model_ly.fit(X,y)
predicted_y0 = model_ly.predict([[0]])[0]
predicted_y1 = model_ly.predict([[1]])[0]
predicted_y2 = model_ly.predict([[2]])[0]
predicted_y0,predicted_y1,predicted_y2

c:\김지은\파이썬_src\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\김지은\파이썬_src\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\김지은\파이썬_src\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


(np.float64(119.83333333333333),
 np.float64(122.33333333333333),
 np.float64(124.83333333333333))

In [23]:
train_df = df.drop(columns=['mpg'])
train_df.head(2)
# cylinders,model year, origin,mpg_qcut --> onehot
# onehot 이후에 onehot에 대상이된 컬럼은 drop
# 제조사컬럼에서 제조사만 추출해서 -> onehot
# 하나의 데이터프레임으로 결합 concat

KeyError: "['mpg'] not found in axis"

In [ ]:
# one하고 예측 hot을 적용해서 학습
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
origin_cols = ['cylinders','model year', 'origin','maker']
train_df['maker'] = [n_list[0] for n_list in df['name'].str.split()]
total_onehots = []
for colname in origin_cols:
    encoder_fit_transform = encoder.fit_transform(train_df[[colname]])
    cols = encoder.get_feature_names_out([colname])
    total_onehots.append(pd.DataFrame.sparse.from_spmatrix(encoder_fit_transform,columns=cols))

In [ ]:
total_onehots.insert(0,train_df)
new_train_df = pd.concat(total_onehots,axis=1)
new_train_df = new_train_df.drop(columns=origin_cols)
new_train_df.head()

In [ ]:
# 범주형 데이터 : 데이터 범위내에서 결정되는 데이터 category
# 연속형 데이터 : 범위가 없는 변화무쌍한 데이터
# 고양이 개 새 -> 범주형 데이터 0과 1로 표현
# 고양이 [1,0,0] 고양이 : on , 개  : off , 새 : on
# 새 [0,0,1]
# 개 [0,1,0]

# 고양이 : 0, 개 : 1, 새 : 2

# 키를 예측 모델, 신발사이즈 변수사용
# 1(작다) / 2(중간) / 3(크다)
# 3번 신발이 신은 사람이 1번신발을 신을 사람보다 신발 사이즈가 3배 증가 --> 아 키도 3배 크다고 인식
# 신발 사이즈 변수 : 카테고리... 모델 학습할때, 참고용도
# https://chatgpt.com/share/68dded47-2648-8012-b5bc-d03b7240f148